# Hello all, I hope you'll find this notebook interesting. With many public notebooks out there, this notebook is **the highest scoring** using logistic regression with CV:0.95551

Let's get started

# Import

I earlier used cross_val_score for cross-validation but then moved to KFold so that OOF predictions can be saved. And changed StandardScaler to MinMaxScaler (improved CV).

In [ ]:
# imports
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression # Model
from sklearn.model_selection import cross_val_score, KFold # Model Evaluation
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore") # ignore, lol

# Load

The column ```id``` is dropped from the dataset as it does not hold any strong signal, but since it is one of the important value needed for submission of predictions, ```test.id``` is preserved.

In [ ]:
# Loading Data
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv') 
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')

# Setting the target - category to 0/1
train['Heart Disease']=train['Heart Disease'].map({'Absence':0, 'Presence':1})
train.drop('id', axis=1, inplace=True) # id is not used for model training and prediction
testID=test.pop('id') # important to preserve test IDs for final submission.
y=train.pop('Heart Disease') # TARGET

train.head(3)

Below we can see the number of unique values in the data

In [ ]:
train.nunique()

Ideally we one-hot encode features with low cardinality, but since the cardinality is not high compared to the amount of data, we can try encoding all. This would generate around 430 features

In [ ]:
BASE_FEATURES=list(train.columns)

# ONE-HOT

In [ ]:
# One Hot All columns 
# (Selective Encoding can be done, but that does not help here)
oh_cols = BASE_FEATURES
oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

oh_train = pd.DataFrame(oh.fit_transform(train[oh_cols]))
train[oh.get_feature_names_out(train[oh_cols].columns)] = oh_train.astype(int) #.astype('category')

oh_test= pd.DataFrame(oh.transform(test[oh_cols]))
test[oh.get_feature_names_out(test[oh_cols].columns)] = oh_test.astype(int) #.astype('category')

Why not One-Hot encode only the categorical features. 

But take a close look on the data, **630K** records. And the max unique values in any column is note more than 150. Perform EDA if you wish, but let me put some real stats, not one-hot encoding features that seem numerical performs led to a poor CV and LB.

**Why does it work?**

Data is huge. Cardinality of each column is very very low compared to the size of data. Though features may be continuous (numeric), still they repeat so many times.

* Detailed explaination is in the comments, check that.

In [ ]:
# Target Encoding
train['y']=y

TE=[]
for col in BASE_FEATURES:
    temp = train.groupby(col).y.agg('median') # Median and Mode are same here
    train[f'{col}_mode'] = train[col].map(temp)
    test[f'{col}_mode'] = test[col].map(temp)
    TE.append(f'{col}_mode')

    temp = train.groupby(col).y.agg('mean') # Call it simple probability
    train[f'{col}_prob'] = train[col].map(temp)
    test[f'{col}_prob'] = test[col].map(temp)
    TE.append(f'{col}_prob')

    temp = train.groupby(col).y.agg('var') # Call it simple probability
    train[f'{col}_var'] = train[col].map(temp)
    test[f'{col}_var'] = test[col].map(temp)
    TE.append(f'{col}_var')

train.pop('y')
print("Target Encoded")

Target encoding is usually recommended to be performed inside the cross validation folds to avoid leakage. But I don't think there is going to be some serious leakage in the code above, if it happens, public LB shall warn us.

In [ ]:
test = test.fillna(0.5) # target encoding led to some NaN values

# Scaling Data

This is an important step because we are using Logistic Regression.

Two benefits are : 

1. Better performance
2. Becomes Faster

In [ ]:
# Speed of Logistic Regression Increases 
# (results shall also increase, but speed is more observable)
#scaler = StandardScaler()
scaler = MinMaxScaler()
scaled_train = pd.DataFrame(scaler.fit_transform(train))
scaled_train.columns = train.columns
scaled_test = pd.DataFrame(scaler.transform(test))
scaled_test.columns = test.columns

In [ ]:
scaled_train.head()

In [ ]:
scaled_train=scaled_train.fillna(-2)
scaled_test=scaled_test.fillna(-2)

# MODEL

Increasing the max iterations could be worth trying because the model does not converge with default parameters.

In [ ]:
model = LogisticRegression(max_iter=1000) # Our Model, simple one - Logistic Regression

In [ ]:
kf = KFold(n_splits=5, shuffle=False)
i=0
oof=pd.Series(index=scaled_train.index)
test_p=[]
for train_index, val_index in kf.split(scaled_train):
    i+=1
    X_t, y_t = scaled_train.iloc[train_index], y.iloc[train_index]
    X_val, y_val = scaled_train.iloc[val_index], y.iloc[val_index]
    model.fit(X_t, y_t)
    
    temp=model.predict_proba(X_val)[:,-1]
    score = roc_auc_score(y_val, temp)
    print(f'Fold {i} \n Score: {score}')
    oof.iloc[val_index] = temp
    
    pred=model.predict_proba(scaled_test)[:,-1]
    test_p.append(pred)

print(f"CV: {roc_auc_score(y, oof)}")

# Final Submission

In [ ]:
model.fit(scaled_train, y)

In [ ]:
out=model.predict_proba(scaled_test)[:,-1]

final = pd.DataFrame({'id':testID, 'Heart Disease':out})
final.to_csv('submission.csv', index=False) # submission file'
print("Submitted Succesfully!!")
final.head()

# Saving OOF predictions

In [ ]:
oof.to_csv('oof_lr.csv', index=False)

# Next Steps

1. Model coefficients and other metrics like mutual info can be used to find the important features and the non-contributing ones can be dropped.
2. New features can be generated using interactions of two or more features, or target encoding can also be performed.
3. Instead of one-hot encoding all, using the first point, selective encoding can be performed.

# This notebook helped? Did it provide some value?

**If yes, consider giving an upvote**

*Have any doubts or suggestions - Do comment*